In [ ]:
workspace = "urban-analytics"
window_start = "2024-01-01"
window_end = "2024-06-30"
openaq_api_key = ""
key_vault_url = ""
openaq_secret_name = "openaq-api-key"


In [ ]:
from datetime import date

from urban_analytics.platform import Platform, Window

platform = Platform.fabric(workspace, Window(date.fromisoformat(window_start), date.fromisoformat(window_end)))


In [ ]:
from urban_analytics.api_sources import (
    AIR_QUALITY_SCHEMA,
    WEATHER_SCHEMA,
    OpenAqClient,
    fetch_air_quality,
    fetch_weather,
    write_bronze,
)


def openaq_key():
    if key_vault_url:
        return notebookutils.credentials.getSecret(key_vault_url, openaq_secret_name)
    return openaq_api_key


weather_rows = write_bronze(spark, fetch_weather(platform.window), platform.bronze.table("openmeteo_daily"), WEATHER_SCHEMA)
air_rows = write_bronze(
    spark,
    fetch_air_quality(OpenAqClient(openaq_key()), platform.window),
    platform.bronze.table("openaq_hourly"),
    AIR_QUALITY_SCHEMA,
)
display(spark.createDataFrame([("openmeteo_daily", weather_rows), ("openaq_hourly", air_rows)], ["table", "rows"]))
